In [ ]:
#Fine tuninig
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/benja/miniconda3/envs/IAmodel/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# Cargamos el modelo que usaremos
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.4.6: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.622 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 291/291 [00:11<00:00, 24.56it/s]
Unsloth: Will load unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit as a legacy tokenizer.


In [4]:
# Usamos LoRa para realizar el fine tune. Lora aplica una tecnica llamada PEFT.

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407, #Investigar
    use_rslora = False,
    loftq_config = None,
)

Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
# Preparamos el dataset de entreno

from datasets import Dataset
import json

with open("./dataset/dataset01.json", "r", encoding="utf-8") as f:
    data = json.load(f)

dataset_hf = Dataset.from_list(data)



# Dividir en train / eval (90% / 10%)
split = dataset_hf.train_test_split(test_size=0.1, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print(f"\nTrain: {len(train_dataset)} ejemplos")
print(f"Eval:  {len(eval_dataset)} ejemplos")


# 6. Guardar en disco (formato Arrow, compatible con Unsloth)
train_dataset.save_to_disk("./dataset_train")
eval_dataset.save_to_disk("./dataset_eval")
print("\nDataset guardado en ./dataset_train y ./dataset_eval")


Map: 100%|██████████| 20/20 [00:00<00:00, 1325.49 examples/s]



Ejemplo formateado:
<｜begin▁of▁sentence｜>Eres Valentina, la asistente virtual de la Secretaría de la carrera de Ingeniería en Computación de la Universidad de La Serena. Atiendes exclusivamente a estudiantes de esa carrera. Tu tono es amable, claro y formal pero cercano. Cuando no tienes información exacta sobre algo,  ...

Train: 18 ejemplos
Eval:  2 ejemplos


Saving the dataset (1/1 shards): 100%|██████████| 2/2 [00:00<00:00, 885.06 examples/s] 


Dataset guardado en ./dataset_train y ./dataset_eval


In [ ]:
# Cargar dataset
from datasets import load_from_disk

train_data = load_from_disk("./dataset_train")
eval_data  = load_from_disk("./dataset_eval")

# DeepSeek-R1-Distill-Llama-8B ya trae su chat template nativo en el tokenizer.
# No lo reemplazamos por llama-3 para no desalinear el entrenamiento y la inferencia.
if tokenizer.chat_template is None:
    raise ValueError("El tokenizer no trae chat_template nativo. Revisa la versión del modelo base.")

def apply_template(examples):
    return {
        "text": tokenizer.apply_chat_template(
            examples["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_data = train_data.map(apply_template)
eval_data  = eval_data.map(apply_template)

print("\nVerificación del template nativo:")
print(train_data[0]["text"])

# Setup entreno
import inspect
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

sft_config_kwargs = dict(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=1,
    optim="adamw_8bit",
    output_dir="outputs",
)

# Compatibilidad entre versiones de TRL/Transformers.
sft_signature = inspect.signature(SFTConfig.__init__).parameters
if "evaluation_strategy" in sft_signature:
    sft_config_kwargs["evaluation_strategy"] = "epoch"
elif "eval_strategy" in sft_signature:
    sft_config_kwargs["eval_strategy"] = "epoch"

if "save_strategy" in sft_signature:
    sft_config_kwargs["save_strategy"] = "epoch"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    dataset_text_field="text",
    max_seq_length= 2048,
    dataset_num_proc=2,
    args=SFTConfig(**sft_config_kwargs),
)

Unsloth: Tokenizing ["text"] (num_proc=10): 100%|██████████| 18/18 [00:12<00:00,  1.45 examples/s]


In [6]:
#Run del entreno
trainer_stats = trainer.train()
# Ver resultados
print(trainer_stats)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,6.298638
2,6.298616
3,6.263473
4,6.302345
5,6.128778
6,5.663044
7,5.439633
8,4.970652
9,4.139380


TrainOutput(global_step=9, training_loss=5.722728888193767, metrics={'train_runtime': 47.8878, 'train_samples_per_second': 1.128, 'train_steps_per_second': 0.188, 'total_flos': 921970640904192.0, 'train_loss': 5.722728888193767, 'epoch': 3.0})


In [ ]:
FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "Eres Valentina, la asistente virtual de la Secretaría de la carrera de Ingeniería en Computación de la Universidad de La Serena."
    },
    {
        "role": "user",
        "content": "necesito reclamar una nota que creo que está mal"
    }
]

# Formateamos primero el texto con el template nativo y luego tokenizamos
# explícitamente para obtener input_ids + attention_mask.
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
    add_special_tokens=False,
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    pad_token_id=tokenizer.eos_token_id,
)

prompt_length = inputs["input_ids"].shape[1]
response_ids = outputs[0][prompt_length:]
response = tokenizer.decode(
    response_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True,
)
print(response)


('secretariaModel-8b-trained-v0.1/tokenizer_config.json',
 'secretariaModel-8b-trained-v0.1/chat_template.jinja',
 'secretariaModel-8b-trained-v0.1/tokenizer.json')

In [8]:
model.save_pretrained_gguf('finetuned_model', tokenizer,quantization_method='q4_k_m', maximum_memory_usage=0.3)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /home/benja/.cache/huggingface/hub


Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 231.69it/s]


Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [00:00<00:00, 64280.52it/s]


...

[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['finetuned_model_gguf/deepseek-r1-distill-llama-8b.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['finetuned_model_gguf/deepseek-r1-distill-llama-8b.Q4_K_M.gguf']
Unsloth: No Ollama template mapping found for model 'unsloth/deepseek-r1-distill-llama-8b'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: /home/benja/.unsloth/llama.cpp/llama-cli --model finetuned_model_gguf/deepseek-r1-distill-llama-8b.Q4_K_M.gguf -p "why is the sky blue?"


{'save_directory': 'finetuned_model',
 'gguf_directory': 'finetuned_model_gguf',
 'gguf_files': ['finetuned_model_gguf/deepseek-r1-distill-llama-8b.Q4_K_M.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': True}